<a href="https://colab.research.google.com/github/Gauravb853/Mini_sales_analysis-/blob/main/Project_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [79]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statistics
from itertools import groupby

In [47]:
customer = pd.read_csv('/content/customers.csv')
orders = pd.read_csv('/content/orders.csv')


In [49]:
customer.head()


,customer_id,signup_date,city
0,1,2023-01-15,New York
1,2,2023-02-20,London
2,3,NaN,Mumbai
3,4,2023-03-05,New York
4,5,2023-01-30,NaN


In [50]:
customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   customer_id  8 non-null      int64 
 1   signup_date  7 non-null      object
 2   city         7 non-null      object
dtypes: int64(1), object(2)
memory usage: 324.0+ bytes


In [52]:
orders.head()

,order_id,customer_id,order_date,order_amount
0,101,1,2023-02-01,250.0
1,102,2,2023-02-25,180.0
2,103,3,2023-03-01,0.0
3,104,1,2023-03-10,320.0
4,105,4,NaN,150.0


In [53]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   order_id      10 non-null     int64  
 1   customer_id   10 non-null     int64  
 2   order_date    9 non-null      object 
 3   order_amount  9 non-null      float64
dtypes: float64(1), int64(2), object(1)
memory usage: 452.0+ bytes


In [54]:
customer.isnull().sum()

,0
customer_id,0
signup_date,1
city,1


In [55]:
orders.isnull().sum()

,0
order_id,0
customer_id,0
order_date,1
order_amount,1


In [68]:
# customers
customer['city'] = customer['city'].fillna('Unknown')

# orders
orders = orders.dropna(subset=['order_date'])
orders = orders[orders['order_amount'] > 0]


In [57]:
customer.head()

,customer_id,signup_date,city
0,1,2023-01-15,New York
1,2,2023-02-20,London
2,3,NaN,Mumbai
3,4,2023-03-05,New York
4,5,2023-01-30,Unknown


In [69]:
orders.head()

,order_id,customer_id,order_date,order_amount
0,101,1,2023-02-01,250.0
1,102,2,2023-02-25,180.0
3,104,1,2023-03-10,320.0
7,108,7,2023-04-15,220.0
8,109,8,2023-04-18,300.0


In [70]:
customer['signup_date'] = pd.to_datetime(customer['signup_date'],format='mixed',errors='coerce')
orders['order_date'] = pd.to_datetime(orders['order_date'],format='mixed',errors='coerce')

In [67]:
customer.dtypes

,0
customer_id,int64
signup_date,datetime64[ns]
city,object


In [66]:
orders.dtypes

,0
order_id,int64
customer_id,int64
order_date,datetime64[ns]
order_amount,float64


In [71]:
orders.isnull().sum()

,0
order_id,0
customer_id,0
order_date,0
order_amount,0


In [72]:
data= orders.merge(
    customer,
    on='customer_id',
    how='left'
)

In [73]:
data.head()

,order_id,customer_id,order_date,order_amount,signup_date,city
0,101,1,2023-02-01,250.0,2023-01-15,New York
1,102,2,2023-02-25,180.0,2023-02-20,London
2,104,1,2023-03-10,320.0,2023-01-15,New York
3,108,7,2023-04-15,220.0,2023-04-10,London
4,109,8,2023-04-18,300.0,2023-04-12,Mumbai


In [74]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   order_id      5 non-null      int64         
 1   customer_id   5 non-null      int64         
 2   order_date    5 non-null      datetime64[ns]
 3   order_amount  5 non-null      float64       
 4   signup_date   5 non-null      datetime64[ns]
 5   city          5 non-null      object        
dtypes: datetime64[ns](2), float64(1), int64(2), object(1)
memory usage: 372.0+ bytes


In [75]:
data['order_amount'].sum()

np.float64(1270.0)

In [76]:
data['order_amount'].mean()

np.float64(254.0)

#Order per customer

In [82]:

data.groupby('customer_id')['order_id'].count()

,order_id
customer_id,
1,2
2,1
7,1
8,1


# Revenue per customer

In [83]:
data.groupby('customer_id')['order_amount'].sum()

,order_amount
customer_id,
1,570.0
2,180.0
7,220.0
8,300.0


#Monthly revenue trend


In [85]:
data['ORDER_MONTH'] = data['order_date'].dt.to_period('M')
data.groupby('ORDER_MONTH')['order_amount'].sum()

,order_amount
ORDER_MONTH,
2023-02,430.0
2023-03,320.0
2023-04,520.0


# One time or Repeater

In [88]:
order_count = data.groupby('customer_id')['order_id'].count()
one_time = (order_count==1).sum()
repeat = (order_count>1).sum()

np.int64(3)

# Top city by revenue

In [90]:
city = data.groupby('city')['order_amount'].sum()
print(city)

city
London      400.0
Mumbai      300.0
New York    570.0
Name: order_amount, dtype: float64
